In [57]:
import geopandas as gpd
import pandas as pd
from glob import glob
import pyarrow as pa

In [16]:
rename_map = {
    'Station ID': 'station_id',
    'Site Name': 'site_name',
    'Address': 'address',
    'Site Type': 'site_type',
    'Station Type': 'station_type',
    'Longitude': 'longitude',
    'Latitude': 'latitude',
    'Start Date': 'start_date',
}

In [40]:
stations = pd.read_csv('./raw/Stations_Locations.csv')
stations = gpd.GeoDataFrame(
    stations, geometry=gpd.points_from_xy(stations.Longitude, stations.Latitude)
)
stations = stations.rename(columns=rename_map)[[*rename_map.values(), 'geometry']]
# stations['station_type'] = stations.groupby(['latitude', 'longitude']).station_type.transform(lambda t: ','.join(t))
# stations['station_type'] = stations.station_type.where(~stations.station_type.str.contains(','), other='BOTH')
# stations = stations.groupby(['latitude', 'longitude']).first().reset_index()

In [ ]:
stations

In [42]:
stations.to_file('./transformed/station_locations.json', driver="GeoJSON")

In [47]:
daily_files = sorted(glob('./raw/daily/2024/*.csv'))

In [65]:
for f in daily_files:
    d = pd.read_csv(f)
    fname = f"../../../static/data/{f.split('/')[-1][:-15]}_daily_2024.arrow"
    schema = pa.Schema.from_pandas(d, preserve_index=False)
    table = pa.Table.from_pandas(d, preserve_index=False)
    writer = pa.ipc.new_file(fname, schema)
    writer.write(table)
    writer.close()

In [67]:
d.columns

Index(['stationID', 'tz', 'obsTimeUtc', 'obsTimeLocal', 'epoch', 'lat', 'lon',
       'solarRadiationHigh', 'uvHigh', 'winddirAvg', 'humidityHigh',
       'humidityLow', 'humidityAvg', 'qcStatus', 'tempHigh', 'tempLow',
       'tempAvg', 'windspeedHigh', 'windspeedLow', 'windspeedAvg',
       'windgustHigh', 'windgustLow', 'windgustAvg', 'dewptHigh', 'dewptLow',
       'dewptAvg', 'windchillHigh', 'windchillLow', 'windchillAvg',
       'heatindexHigh', 'heatindexLow', 'heatindexAvg', 'pressureMax',
       'pressureMin', 'pressureTrend', 'precipRate', 'precipTotal'],
      dtype='object')